<a href="https://colab.research.google.com/github/Osondu-ifunanya/Illegal-logging-detection-using-object-detection-on-Planet-imagery/blob/main/illegal%20logging%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Illegal Logging Detection using YOLOv8 (Synthetic Planet Imagery)

Workflow:
1. Generate synthetic satellite images
2. Create bounding box labels for logging areas
3. Train YOLOv8 model
4. Perform detection
"""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

np.random.seed(42)

# ------------------------------------------
# 1. Create Synthetic Dataset
# ------------------------------------------

def create_dataset(base_path="dataset", n_images=50, size=256):
    os.makedirs(f"{base_path}/images/train", exist_ok=True)
    os.makedirs(f"{base_path}/labels/train", exist_ok=True)

    for i in range(n_images):
        img = np.zeros((size, size, 3), dtype=np.uint8)

        # forest background (green texture)
        img[:,:,1] = np.random.randint(100, 200, (size, size))
        img[:,:,0] = np.random.randint(20, 80, (size, size))
        img[:,:,2] = np.random.randint(20, 80, (size, size))

        # add logging patches (brown rectangles)
        n_logs = np.random.randint(1, 4)
        labels = []

        for _ in range(n_logs):
            x1 = np.random.randint(20, size-60)
            y1 = np.random.randint(20, size-60)
            w = np.random.randint(20, 60)
            h = np.random.randint(20, 60)

            x2 = x1 + w
            y2 = y1 + h

            cv2.rectangle(img, (x1,y1), (x2,y2), (139,69,19), -1)

            # YOLO format
            xc = (x1 + x2) / 2 / size
            yc = (y1 + y2) / 2 / size
            bw = w / size
            bh = h / size

            labels.append(f"0 {xc} {yc} {bw} {bh}")

        # save image
        cv2.imwrite(f"{base_path}/images/train/img_{i}.jpg", img)

        # save label
        with open(f"{base_path}/labels/train/img_{i}.txt", "w") as f:
            f.write("\n".join(labels))

create_dataset()

# ------------------------------------------
# 2. Create YAML Config
# ------------------------------------------

yaml_content = """
path: dataset
train: images/train
val: images/train

names:
  0: logging
"""

with open("data.yaml", "w") as f:
    f.write(yaml_content)

# ------------------------------------------
# 3. Train YOLO Model
# ------------------------------------------

model = YOLO("yolov8n.pt")

model.train(
    data="data.yaml",
    epochs=10,
    imgsz=256,
    batch=8
)

# ------------------------------------------
# 4. Run Detection
# ------------------------------------------

results = model.predict("dataset/images/train/img_0.jpg", conf=0.3)

# ------------------------------------------
# 5. Visualize Result
# ------------------------------------------

img = cv2.imread("dataset/images/train/img_0.jpg")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

for r in results:
    for box in r.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cv2.rectangle(img, (x1,y1), (x2,y2), (255,0,0), 2)

plt.imshow(img)
plt.title("Detected Illegal Logging Areas")
plt.axis("off")
plt.show()